# Section A : Given fault path and implementing gates, output the two logical errors and the syndromes

In [31]:
import numpy as np
from itertools import product

# ---------------------------------------------------------------------------
# Pauli algebra — binary symplectic representation
# ---------------------------------------------------------------------------
# Single-qubit Pauli: (x, z) in F_2^2
#   (0,0)=I  (1,0)=X  (1,1)=Y  (0,1)=Z
# Multiplication (mod phase): XOR componentwise

PAULI_LABEL   = {(0,0):'I', (1,0):'X', (1,1):'Y', (0,1):'Z'}
PAULI_FROM_CH = {'I':(0,0), 'X':(1,0), 'Y':(1,1), 'Z':(0,1)}
LOGICAL_NAMES = ['I', 'X', 'Y', 'Z']

def pauli_mul(xv1, zv1, xv2, zv2):
    return xv1 ^ xv2, zv1 ^ zv2

def pauli_anticommutes(xv1, zv1, xv2, zv2):
    return bool((np.dot(xv1, zv2) + np.dot(zv1, xv2)) % 2)

def pauli_weight(xv, zv):
    return int(np.sum(xv | zv))

def pauli_str(xv, zv):
    return ''.join(PAULI_LABEL[(int(xv[i]), int(zv[i]))] for i in range(len(xv)))

def str_to_pauli(s):
    s = s.upper()
    assert len(s) == 5, "Need exactly 5 characters"
    xv = np.array([PAULI_FROM_CH[c][0] for c in s], dtype=int)
    zv = np.array([PAULI_FROM_CH[c][1] for c in s], dtype=int)
    return xv, zv

def all_paulis(n):
    out = []
    for bits in product([0,1], repeat=2*n):
        xv = np.array(bits[0::2], dtype=int)
        zv = np.array(bits[1::2], dtype=int)
        out.append((xv, zv))
    return out

# ---------------------------------------------------------------------------
# [[5,1,3]] stabilizer code
# ---------------------------------------------------------------------------
N = 5

STAB_GENS = [
    (np.array([1,0,0,1,0]), np.array([0,1,1,0,0])),  # g1: XZZXI
    (np.array([0,1,0,0,1]), np.array([0,0,1,1,0])),  # g2: IXZZX
    (np.array([1,0,1,0,0]), np.array([0,0,0,1,1])),  # g3: XIXZZ
    (np.array([0,1,0,1,0]), np.array([1,0,0,0,1])),  # g4: ZXIXZ
]
XLOG = (np.array([1,1,1,1,1]), np.array([0,0,0,0,0]))  # X_L = XXXXX
ZLOG = (np.array([0,0,0,0,0]), np.array([1,1,1,1,1]))  # Z_L = ZZZZZ

def compute_syndrome(xv, zv):
    s = 0
    for i, (gx, gz) in enumerate(STAB_GENS):
        bit = (np.dot(xv, gz) + np.dot(zv, gx)) % 2
        s |= (int(bit) << (3 - i))
    return s

def syndrome_bits(s):
    return [(s >> (3-i)) & 1 for i in range(4)]

def build_corrections():
    best = {}
    for xv, zv in all_paulis(N):
        s = compute_syndrome(xv, zv)
        w = pauli_weight(xv, zv)
        if s not in best or w < best[s][0]:
            best[s] = (w, xv.copy(), zv.copy())
    return {s: (xv, zv) for s, (w, xv, zv) in best.items()}

CORRECTIONS = build_corrections()

def logical_class(xv, zv):
    """Returns (idx, (x_log, z_log))  idx: 0=I 1=X 2=Y 3=Z"""
    x_log = int(pauli_anticommutes(xv, zv, *ZLOG))
    z_log = int(pauli_anticommutes(xv, zv, *XLOG))
    idx = {(0,0):0, (1,0):1, (1,1):2, (0,1):3}[(x_log, z_log)]
    return idx, (x_log, z_log)

def logical_mul(l1_bits, l2_bits):
    """Multiply two logical Paulis given as (x_log, z_log) tuples. Returns (idx, bits)."""
    bits = (l1_bits[0] ^ l2_bits[0], l1_bits[1] ^ l2_bits[1])
    idx  = {(0,0):0, (1,0):1, (1,1):2, (0,1):3}[bits]
    return idx, bits

print("[[5,1,3]] setup loaded.")
print("Stabilizer generators:", [pauli_str(gx, gz) for gx, gz in STAB_GENS])
print("X_L =", pauli_str(*XLOG), "   Z_L =", pauli_str(*ZLOG))

[[5,1,3]] setup loaded.
Stabilizer generators: ['XZZXI', 'IXZZX', 'XIXZZ', 'ZXIXZ']
X_L = XXXXX    Z_L = ZZZZZ


In [32]:
# ============================================================
#  >>> INPUTS: Set the fault path here <<<
# ============================================================
#
# Circuit layout (truncated exRec, back-to-front analysis):
#
#   P1 - FTEC_a - P2 - [Z gate] - P3 - * - FTEC_b - P4 - [X gate] - P5 - FTEC_c - P6
#
#   FTEC_a : leading EC of Z exRec  — corrects syndrome(P1)
#   *      : star decoder (between P3 and FTEC_b) — captures s_in for X exRec
#   FTEC_b : leading EC of X exRec  — corrects the syndrome injected by the star decoder
#   FTEC_c : trailing EC of X exRec
#
# Transversal Z = ZZZZZ and X = XXXXX act trivially on Pauli errors in the
# binary symplectic representation (they only introduce phases, which are ignored).
#
# Each Pi is a 5-character string with characters in {I, X, Y, Z}.

P1 = 'IIIII'   # fault before FTEC_a  (sets incoming syndrome of Z exRec)
P2 = 'YIXII'   # fault between FTEC_a and Z gate
P3 = 'IIIII'   # fault between Z gate and star decoder (*)
P4 = 'IIIII'   # fault between FTEC_b and X gate
P5 = 'IIIII'   # fault between X gate and FTEC_c
P6 = 'YIXII'   # fault after FTEC_c (output data error, not corrected)
# ============================================================

In [33]:
# ============================================================
#  Z exRec analysis
#  Structure: P1 - FTEC_a - P2 - [Z] - P3 - (star decoder boundary)
# ============================================================

xP1, zP1 = str_to_pauli(P1)
xP2, zP2 = str_to_pauli(P2)
xP3, zP3 = str_to_pauli(P3)

# --- Step 1: FTEC_a sees syndrome(P1) and applies recovery U_Z = Q_{syndrome(P1)}
s_P1   = compute_syndrome(xP1, zP1)
Ux, Uz = CORRECTIONS[s_P1]          # U_Z: the recovery that FTEC_a applies
U_Z_str = pauli_str(Ux, Uz)

# --- Step 2: After FTEC_a correction — P1 · U_Z has syndrome 0, possible logical residual
xAft, zAft = pauli_mul(xP1, zP1, Ux, Uz)   # P1 · U_Z  (syndrome 0)

# --- Step 3: P2 then Z gate (trivial in symplectic) then P3 accumulate
#     E_star = P1·U_Z · P2 · P3   (total error at the star decoder position)
xE_star, zE_star = pauli_mul(xAft, zAft, xP2, zP2)
xE_star, zE_star = pauli_mul(xE_star, zE_star, xP3, zP3)

# --- Step 4: Syndrome at star decoder → becomes s_in for X exRec
s_in_X = compute_syndrome(xE_star, zE_star)
# (Note: syndrome(P1·U_Z) = 0, so s_in_X = syndrome(P2·P3) only)

# --- Step 5: Virtual correction at the star decoder gives L_Z
#     (This is what a trailing EC *would* apply if the Z exRec had one.)
Qvx, Qvz = CORRECTIONS[s_in_X]
xResZ, zResZ = pauli_mul(Qvx, Qvz, xE_star, zE_star)   # Q_{s_in_X} · E_star
L_Z_idx, L_Z_bits = logical_class(xResZ, zResZ)

# --- Display
sep = '=' * 55
print(sep)
print("  Z exRec  (P1 - FTEC_a - P2 - Z - P3 - *)")
print(sep)
print()
print(f"  P1 (fault before FTEC_a)   = {P1.upper()}")
print(f"  syndrome(P1)               = {s_P1}  {syndrome_bits(s_P1)}")
print()
print(f"  U_Z = Q_{{s_P1}} (FTEC_a recovery) = {U_Z_str}")
print(f"  (weight {pauli_weight(Ux, Uz)}: FTEC_a applies this correction)")
print()
print(f"  P1 · U_Z (residual after FTEC_a) = {pauli_str(xAft, zAft)}")
print(f"    → syndrome 0, logical class = {LOGICAL_NAMES[logical_class(xAft, zAft)[0]]}")
print()
print(f"  P2  = {P2.upper()}  (fault between FTEC_a and Z gate)")
print(f"  P3  = {P3.upper()}  (fault between Z gate and star decoder)")
print(f"  [Z gate is transversal — trivial in symplectic, errors pass through unchanged]")
print()
print(f"  E_star = P1·U_Z·P2·P3 = {pauli_str(xE_star, zE_star)}")
print(f"  (total accumulated error at the star decoder position)")
print()
print(f"  s_in_X = syndrome(E_star) = {s_in_X}  {syndrome_bits(s_in_X)}")
print(f"  (= syndrome(P2·P3) since P1·U_Z has syndrome 0)")
print()
print(f"  Virtual trailing correction Q_{{s_in_X}} = {pauli_str(Qvx, Qvz)}")
print(f"  E_residual_Z = Q_{{s_in_X}} · E_star = {pauli_str(xResZ, zResZ)}")
print()
print(f"  *** L_Z (logical error from Z exRec) = {LOGICAL_NAMES[L_Z_idx]} ***")
print(sep)

  Z exRec  (P1 - FTEC_a - P2 - Z - P3 - *)

  P1 (fault before FTEC_a)   = IIIII
  syndrome(P1)               = 0  [0, 0, 0, 0]

  U_Z = Q_{s_P1} (FTEC_a recovery) = IIIII
  (weight 0: FTEC_a applies this correction)

  P1 · U_Z (residual after FTEC_a) = IIIII
    → syndrome 0, logical class = I

  P2  = YIXII  (fault between FTEC_a and Z gate)
  P3  = IIIII  (fault between Z gate and star decoder)
  [Z gate is transversal — trivial in symplectic, errors pass through unchanged]

  E_star = P1·U_Z·P2·P3 = YIXII
  (total accumulated error at the star decoder position)

  s_in_X = syndrome(E_star) = 7  [0, 1, 1, 1]
  (= syndrome(P2·P3) since P1·U_Z has syndrome 0)

  Virtual trailing correction Q_{s_in_X} = IIIIY
  E_residual_Z = Q_{s_in_X} · E_star = YIXIY

  *** L_Z (logical error from Z exRec) = X ***


In [34]:
# ============================================================
#  X exRec analysis  (full exRec with leading EC)
#  Structure:  *_{s_in_X} - FTEC_b - P4 - [X] - P5 - FTEC_c - P6
# ============================================================
# FTEC_b is the leading EC: sees s_in_X from the star decoder, applies U_X = Q_{s_in_X},
# leaving the state at syndrome 0.
# E_fault_X = P4·P5·P6 — all faults after FTEC_b are seen by FTEC_c and corrected.

xP4, zP4 = str_to_pauli(P4)
xP5, zP5 = str_to_pauli(P5)
xP6, zP6 = str_to_pauli(P6)

# U_X: recovery that FTEC_b applies
Ux2, Uz2 = CORRECTIONS[s_in_X]
U_X_str  = pauli_str(Ux2, Uz2)

# E_fault_X = P4 · P5 · P6  (X gate trivial in symplectic)
xEfX, zEfX = pauli_mul(xP4, zP4, xP5, zP5)
xEfX, zEfX = pauli_mul(xEfX, zEfX, xP6, zP6)

# s_out_X: syndrome measured by FTEC_c
s_out_X = compute_syndrome(xEfX, zEfX)

# FTEC_c correction → E_residual_X (syndrome 0)
Qox, Qoz   = CORRECTIONS[s_out_X]
xResX, zResX = pauli_mul(Qox, Qoz, xEfX, zEfX)
L_X_idx, L_X_bits = logical_class(xResX, zResX)

# --- Display
sep = '=' * 55
print(sep)
print("  X exRec  (* - FTEC_b - P4 - X - P5 - FTEC_c - P6)")
print(sep)
print()
print(f"  s_in_X (from Z exRec star decoder) = {s_in_X}  {syndrome_bits(s_in_X)}")
print()
print(f"  U_X = Q_{{s_in_X}} (FTEC_b recovery) = {U_X_str}")
print(f"  (weight {pauli_weight(Ux2, Uz2)}: FTEC_b corrects incoming syndrome → state syndrome 0)")
print()
print(f"  P4 = {P4.upper()}  (fault between FTEC_b and X gate)")
print(f"  P5 = {P5.upper()}  (fault between X gate and FTEC_c)")
print(f"  P6 = {P6.upper()}  (fault after FTEC_c)")
print(f"  [X gate is transversal — trivial in symplectic]")
print()
print(f"  E_fault_X = P4·P5·P6 = {pauli_str(xEfX, zEfX)}")
print()
print(f"  s_out_X = syndrome(E_fault_X) = {s_out_X}  {syndrome_bits(s_out_X)}")
print()
print(f"  FTEC_c correction Q_{{s_out_X}} = {pauli_str(Qox, Qoz)}")
print(f"  E_residual_X = Q_{{s_out_X}}·E_fault_X = {pauli_str(xResX, zResX)}")
print()
print(f"  *** L_X (logical error from X exRec) = {LOGICAL_NAMES[L_X_idx]} ***")
print(sep)

  X exRec  (* - FTEC_b - P4 - X - P5 - FTEC_c - P6)

  s_in_X (from Z exRec star decoder) = 7  [0, 1, 1, 1]

  U_X = Q_{s_in_X} (FTEC_b recovery) = IIIIY
  (weight 1: FTEC_b corrects incoming syndrome → state syndrome 0)

  P4 = IIIII  (fault between FTEC_b and X gate)
  P5 = IIIII  (fault between X gate and FTEC_c)
  P6 = YIXII  (fault after FTEC_c)
  [X gate is transversal — trivial in symplectic]

  E_fault_X = P4·P5·P6 = YIXII

  s_out_X = syndrome(E_fault_X) = 7  [0, 1, 1, 1]

  FTEC_c correction Q_{s_out_X} = IIIIY
  E_residual_X = Q_{s_out_X}·E_fault_X = YIXIY

  *** L_X (logical error from X exRec) = X ***


In [35]:
# ============================================================
#  Summary
# ============================================================

sep2 = '*' * 55
print(sep2)
print("  CIRCUIT SUMMARY")
print(f"  P1 - FTEC_a - P2 - [Z] - P3 - * - FTEC_b - P4 - [X] - P5 - FTEC_c - P6")
print(sep2)
print()
print(f"  Z exRec  (P1 - FTEC_a - P2 - [Z] - P3 - *):")
print(f"    U_Z     = {U_Z_str}  (FTEC_a recovery)")
print(f"    s_in_X  = {s_in_X}  {syndrome_bits(s_in_X)}  (star decoder output)")
print(f"    L_Z     = {LOGICAL_NAMES[L_Z_idx]}")
print()
print(f"  X exRec  (* - FTEC_b - P4 - [X] - P5 - FTEC_c - P6):")
print(f"    U_X     = {U_X_str}  (FTEC_b leading EC recovery)")
print(f"    s_out_X = {s_out_X}  {syndrome_bits(s_out_X)}  (FTEC_c output syndrome)")
print(f"    L_X     = {LOGICAL_NAMES[L_X_idx]}")
print(sep2)

*******************************************************
  CIRCUIT SUMMARY
  P1 - FTEC_a - P2 - [Z] - P3 - * - FTEC_b - P4 - [X] - P5 - FTEC_c - P6
*******************************************************

  Z exRec  (P1 - FTEC_a - P2 - [Z] - P3 - *):
    U_Z     = IIIII  (FTEC_a recovery)
    s_in_X  = 7  [0, 1, 1, 1]  (star decoder output)
    L_Z     = X

  X exRec  (* - FTEC_b - P4 - [X] - P5 - FTEC_c - P6):
    U_X     = IIIIY  (FTEC_b leading EC recovery)
    s_out_X = 7  [0, 1, 1, 1]  (FTEC_c output syndrome)
    L_X     = X
*******************************************************


# Section B : Enumerate fault paths, upto a certain power